# 03 — Analytics: PageRank, Bus Factor, Orphan Risk, Sentinel Score

**Goal:** Compute the four core metrics that make this project novel, then combine them into the **Sentinel Score** — a single number expressing how dangerous the failure of each package would be.

## Metrics summary

| Metric | Question answered | Algorithm |
|---|---|---|
| **PageRank** | How central is this package to the ecosystem? | GraphFrames PageRank |
| **Community** | Which cluster of projects is this package part of? | Label Propagation |
| **Jaccard** | Which packages share the same maintainer pool? | Custom aggregation |
| **Bus Factor** | How many active humans maintain this? | Commit window aggregation |
| **Orphan Risk** | Is this package effectively abandoned? | npm registry recency × maintainer count |
| **Sentinel Score** | Combined structural + human risk | PageRank × (1/BF) × recency_decay |

## What this notebook produces
| Output | Contents |
|---|---|
| `processed/pagerank.parquet` | PageRank score per package |
| `processed/communities.parquet` | Community label per node |
| `processed/bus_factor.parquet` | Bus Factor per package |
| `processed/orphan_risk.parquet` | Orphan Risk score + flag |
| `processed/sentinel_scores.parquet` | **Final combined risk table** |

## 0 — Setup

In [ ]:
!pip install -q pyspark==3.5.0 graphframes pandas pyarrow

In [ ]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages graphframes:graphframes:0.8.3-spark3.5-s_2.12 pyspark-shell'
)

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, lower, when, datediff, current_date, countDistinct,
    sum as spark_sum, max as spark_max, array_intersect, size,
    coalesce, greatest, exp, log
)
from pyspark.sql.types import DoubleType

spark = (
    SparkSession.builder
    .appName('BlastRadius-Analytics')
    .config('spark.driver.memory', '20g')
    .config('spark.sql.shuffle.partitions', '50')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

from google.colab import drive
drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/BlastRadius'
USE_SAMPLE = True   # flip to False for full-scale run
DATA_DIR = f'{BASE}/data/sample' if USE_SAMPLE else f'{BASE}/data/processed'
OUT_DIR  = f'{BASE}/data/processed'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Spark ready. USE_SAMPLE={USE_SAMPLE}')

In [ ]:
import pandas as pd
from graphframes import GraphFrame

suffix = '_10k' if USE_SAMPLE else ''

vertices_pd = pd.read_parquet(f'{DATA_DIR}/graph_vertices{suffix}.parquet')
edges_pd    = pd.read_parquet(f'{DATA_DIR}/graph_edges{suffix}.parquet')

vertices = spark.createDataFrame(vertices_pd)
edges    = spark.createDataFrame(edges_pd)
g        = GraphFrame(vertices, edges)

print(f'Graph loaded: {g.vertices.count():,} nodes, {g.edges.count():,} edges')

## 1 — PageRank

**Why PageRank?** Originally designed for web pages, PageRank answers: *"Which nodes get the most link weight from other important nodes?"* In our dependency graph, a package with high PageRank is depended on by many other important packages — not just by leaf packages. This is a much better measure of centrality than raw inbound-edge count.

**Parameters:**
- `resetProbability=0.15` — the standard "teleportation" probability (chance a random walk jumps to a random node). 0.15 is the classic Google value.
- `maxIter=10` — for 10k nodes, 10 iterations is enough for convergence. Increase to 20 for the full 50k graph.

**Important:** We only run PageRank on the package→package dependency subgraph. We exclude contributor nodes because we want to rank packages, not people.

In [ ]:
# Build a package-only subgraph for PageRank
pkg_vertices = g.vertices.filter(col('node_type') == 'package')
dep_edges    = g.edges.filter(col('edge_type') == 'DEPENDS_ON')
g_pkg        = GraphFrame(pkg_vertices, dep_edges)

print('Running PageRank (this takes 1-3 minutes)...')
pr_result = g_pkg.pageRank(resetProbability=0.15, maxIter=10)

pagerank_df = (
    pr_result.vertices
    .select(
        col('id'),
        col('name'),
        col('pagerank'),
        col('dependent_packages'),
        col('stars'),
        col('last_publish'),
        col('github_slug')
    )
    .orderBy(col('pagerank').desc())
)

print('Top 15 packages by PageRank:')
pagerank_df.show(15, truncate=False)

In [ ]:
pagerank_pd = pagerank_df.toPandas()
pagerank_pd.to_parquet(f'{OUT_DIR}/pagerank.parquet', index=False)
print(f'Saved PageRank scores for {len(pagerank_pd):,} packages')

## 2 — Community Detection (Label Propagation)

**Why community detection?** The NPM ecosystem isn't one homogeneous blob — it's a set of overlapping clusters: React ecosystem, testing frameworks, build tools, etc. Community detection finds these clusters automatically.

**Label Propagation algorithm:** Each node starts with its own unique label. In each iteration, every node adopts the most common label among its neighbors. After several iterations, nodes that are densely connected share the same label — they form a community.

**Why not Louvain?** Louvain is more accurate but isn't natively in GraphFrames. Label Propagation is the built-in alternative and works well for large graphs. We document Louvain as a future improvement.

**What we do with this:** In the report, we visualize communities as color-coded clusters in a force-directed graph, and use them to contextualize the Sentinel Score (e.g., "this package is in the React cluster, which means its blast radius is primarily frontend apps").

In [ ]:
print('Running Label Propagation community detection...')
communities_result = g_pkg.labelPropagation(maxIter=5)

communities_df = (
    communities_result
    .select(col('id'), col('name'), col('label').alias('community_id'))
)

# Show the 10 largest communities
print('Top 10 communities by size:')
(
    communities_df
    .groupBy('community_id')
    .count()
    .orderBy(col('count').desc())
    .show(10)
)

# Show a sample of packages from the largest community
top_community = (
    communities_df
    .groupBy('community_id')
    .count()
    .orderBy(col('count').desc())
    .first()['community_id']
)
print(f'\nSample packages from community {top_community}:')
communities_df.filter(col('community_id') == top_community).show(15, truncate=False)

In [ ]:
communities_pd = communities_df.toPandas()
communities_pd.to_parquet(f'{OUT_DIR}/communities.parquet', index=False)
print(f'Saved community labels for {len(communities_pd):,} nodes')

## 3 — Bus Factor

**Why Bus Factor?** The Bus Factor (also called Truck Factor) of a project is: *"How many key maintainers could be hit by a bus before the project collapses?"* A Bus Factor of 1 means the project is entirely dependent on one person. For high-PageRank packages, a Bus Factor of 1 is a critical supply chain risk.

**How we calculate it:** We count the number of distinct GitHub users who committed to a repo's main branch **within the last 12 months**. We use a 12-month window because a maintainer who last contributed 3 years ago is not reliably available.

**Limitation:** This counts people who *committed*, not people who have *npm publish rights*. The npm registry `maintainer_count` (from notebook 01) is a better measure of publish access, but commit frequency is a better measure of active engagement. We use both.

**Recency decay:** We weight each commit by a simple exponential decay: recent commits count fully, older commits count less. This captures that a maintainer who was active 11 months ago is more reliable than one who was last active 11.9 months ago.

In [ ]:
# Load commit data
commits_pd = pd.read_parquet(f'{DATA_DIR}/gh_commits{suffix}.parquet')
commits_sdf = spark.createDataFrame(commits_pd)

# Join commits → package names via github_slug
pkg_slugs = (
    g.vertices
    .filter(col('node_type') == 'package')
    .select(col('name').alias('package_name'), col('github_slug'))
    .filter(col('github_slug').isNotNull())
)

commits_with_pkg = (
    commits_sdf
    .join(pkg_slugs, 'github_slug')
    .withColumn('days_since_commit', datediff(current_date(), col('commit_date')))
)

# Bus Factor = count of distinct active maintainers (commits within 365 days)
ACTIVE_WINDOW_DAYS = 365

bus_factor_df = (
    commits_with_pkg
    .filter(col('days_since_commit') <= ACTIVE_WINDOW_DAYS)
    .groupBy('package_name')
    .agg(
        countDistinct('author_login').alias('bus_factor'),
        spark_max('commit_date').alias('last_commit_date'),
        spark_sum('push_count').alias('total_commits_12mo')
    )
    .withColumn('days_since_last_commit',
        datediff(current_date(), col('last_commit_date')))
    # Recency decay: e^(-days/365) → 1.0 if committed today, ~0.37 if 1 year ago
    .withColumn('recency_decay',
        exp(-col('days_since_last_commit').cast(DoubleType()) / 365.0))
)

print(f'Bus Factor computed for {bus_factor_df.count():,} packages')
print('\nBus Factor distribution:')
bus_factor_df.groupBy('bus_factor').count().orderBy('bus_factor').show(10)

In [ ]:
# For packages with NO commits in our dataset, default Bus Factor = 1 (worst case)
# and recency_decay = 0 (completely stale)
# We'll join later; missing = assumed abandoned

bus_factor_pd = bus_factor_df.toPandas()
bus_factor_pd.to_parquet(f'{OUT_DIR}/bus_factor.parquet', index=False)
print(f'Saved Bus Factor data for {len(bus_factor_pd):,} packages')

print('\nTop 10 packages with Bus Factor = 1 (most critical):')
bf1 = bus_factor_pd[bus_factor_pd['bus_factor'] == 1].nlargest(10, 'total_commits_12mo')
print(bf1[['package_name', 'bus_factor', 'last_commit_date', 'total_commits_12mo']].to_string(index=False))

## 4 — Orphan Risk

**Why a separate Orphan Risk metric?** Bus Factor measures GitHub commit activity. But some packages have:
- Zero recent GitHub commits (maintainer stopped contributing)
- But still only 1-2 npm publish accounts (the only people who can release security patches)

The Orphan Risk score captures this from the npm registry side. A high Orphan Risk package is: widely depended upon, with few publish-rights holders, and hasn't been updated recently. The classic example is `event-stream` — it had 1 maintainer who handed the package to a malicious actor.

**Orphan Score formula:**
```
orphan_score = log(dependent_packages + 1) × (months_since_publish / 12) × (1 / maintainer_count)
```
- `log(dependent_packages + 1)` — logarithmic scale so a package with 10k dependents doesn't completely dominate
- `months_since_publish / 12` — staleness grows linearly; divide by 12 so 1 year = weight of 1
- `1 / maintainer_count` — solo maintainer = full weight; 10 maintainers = 1/10 weight

In [ ]:
import json

maintainers_pd = pd.read_parquet(f'{BASE}/data/processed/npm_maintainers.parquet')
maintainers_pd['npm_maintainers'] = maintainers_pd['npm_maintainers'].apply(json.loads)

maintainers_sdf = spark.createDataFrame(maintainers_pd.drop(columns=['npm_maintainers']))

# Join with package metadata to get dependent_packages count
pkg_meta = (
    g.vertices
    .filter(col('node_type') == 'package')
    .select(
        col('name').alias('pkg_name'),
        coalesce(col('dependent_packages'), lit(0)).alias('dependent_packages')
    )
)

orphan_risk_df = (
    maintainers_sdf
    .join(pkg_meta, col('name') == col('pkg_name'), 'left')
    .withColumn('months_since_publish',
        datediff(current_date(), col('npm_last_modified')) / 30.0)
    .withColumn('orphan_score',
        # log(1 + dependents) × staleness_years × (1 / maintainer_count)
        log(col('dependent_packages').cast(DoubleType()) + 1.0)
        * (col('months_since_publish') / 12.0)
        * (1.0 / greatest(col('maintainer_count').cast(DoubleType()), lit(1.0)))
    )
    .withColumn('is_orphan_risk',
        (col('maintainer_count') <= 2) | (col('months_since_publish') > 12)
    )
    .select(
        'name', 'maintainer_count', 'npm_last_modified',
        'months_since_publish', 'dependent_packages',
        'orphan_score', 'is_orphan_risk', 'latest_version'
    )
    .orderBy(col('orphan_score').desc())
)

print('Top 15 packages by Orphan Risk score:')
orphan_risk_df.show(15, truncate=False)

In [ ]:
orphan_risk_pd = orphan_risk_df.toPandas()
orphan_risk_pd.to_parquet(f'{OUT_DIR}/orphan_risk.parquet', index=False)

flagged = orphan_risk_pd[orphan_risk_pd['is_orphan_risk']]
print(f'Total packages analyzed: {len(orphan_risk_pd):,}')
print(f'Flagged as Orphan Risk:  {len(flagged):,}')
print(f'  - Solo maintainer (≤2): {(orphan_risk_pd["maintainer_count"] <= 2).sum():,}')
print(f'  - Stale (>12 months):   {(orphan_risk_pd["months_since_publish"] > 12).sum():,}')

## 5 — Jaccard Similarity: Contributor Overlap

**Why Jaccard?** Two packages that share the same small pool of maintainers have a hidden coupling: if one maintainer goes rogue or is compromised, both packages are at risk. This pattern was exploited in the `ua-parser-js` attack (same maintainer, multiple packages hijacked at once).

**Jaccard similarity** = |A ∩ B| / |A ∪ B| — the fraction of maintainers shared between two packages. A score of 1.0 means identical maintainer sets; 0.0 means no overlap.

We compute this for high-PageRank packages only (top 500) — computing all pairwise similarities for 10k packages would be O(n²) and impractical.

In [ ]:
from pyspark.sql.functions import collect_set, size, array_intersect, array_union

# Get maintainer sets per package (from MAINTAINS edges in the graph)
# contributor node IDs look like 'user:gaearon' — extract just the login
maintainer_sets = (
    g.edges
    .filter(col('edge_type') == 'MAINTAINS')
    .groupBy(col('dst').alias('pkg_id'))
    .agg(collect_set('src').alias('maintainer_set'))
    .withColumn('n_maintainers', size(col('maintainer_set')))
)

# Limit to top 500 packages by PageRank for pairwise comparison
top500_ids = (
    pagerank_df
    .orderBy(col('pagerank').desc())
    .limit(500)
    .select(col('id').alias('pkg_id'))
)

top500_maintainers = maintainer_sets.join(top500_ids, 'pkg_id')
print(f'Packages with maintainer data in top 500: {top500_maintainers.count():,}')

In [ ]:
# Self-join to compute pairwise Jaccard
# This is O(n²) — manageable for 500 packages
a = top500_maintainers.alias('a')
b = top500_maintainers.alias('b')

jaccard_df = (
    a.crossJoin(b)
    # Only compute upper triangle (avoid duplicates and self-pairs)
    .filter(col('a.pkg_id') < col('b.pkg_id'))
    .withColumn('intersection', size(array_intersect(col('a.maintainer_set'), col('b.maintainer_set'))))
    .withColumn('union_size',   size(array_union(col('a.maintainer_set'), col('b.maintainer_set'))))
    .withColumn('jaccard', col('intersection').cast(DoubleType()) / col('union_size').cast(DoubleType()))
    .filter(col('jaccard') > 0.0)   # only pairs with any overlap
    .select(
        col('a.pkg_id').alias('pkg_a'),
        col('b.pkg_id').alias('pkg_b'),
        col('intersection'),
        col('union_size'),
        col('jaccard')
    )
    .orderBy(col('jaccard').desc())
)

print(f'Package pairs with shared maintainers: {jaccard_df.count():,}')
print('\nTop 15 most similar package pairs (by maintainer overlap):')
(
    jaccard_df
    .join(g.vertices.select(col('id'), col('name').alias('name_a')), col('pkg_a') == col('id')).drop('id')
    .join(g.vertices.select(col('id'), col('name').alias('name_b')), col('pkg_b') == col('id')).drop('id')
    .select('name_a', 'name_b', 'intersection', 'jaccard')
    .show(15, truncate=False)
)

In [ ]:
jaccard_pd = jaccard_df.toPandas()
jaccard_pd.to_parquet(f'{OUT_DIR}/jaccard.parquet', index=False)
print(f'Saved {len(jaccard_pd):,} Jaccard pairs')

## 6 — Sentinel Score (the main output)

**Formula:**
```
sentinel_score = pagerank × (1 / bus_factor) × recency_decay
```

**Interpretation:**
- High `pagerank` → the package is structurally central (many important things depend on it)
- Low `bus_factor` → few humans actively maintain it
- Low `recency_decay` → the last commit was a long time ago

A package that is **central + solo-maintained + abandoned** gets the highest score — and is the most dangerous point of failure in the ecosystem.

**Default values for missing data:**
- If no commit data: `bus_factor = 1`, `recency_decay = 0.01` (effectively abandoned)
- If pagerank is missing: `pagerank = 0` (package not in graph center)

In [ ]:
from pyspark.sql.functions import broadcast

# Reload bus_factor as Spark DF from its saved parquet
bus_factor_sdf = spark.createDataFrame(bus_factor_pd)

# Join PageRank + Bus Factor + Community
communities_sdf = spark.createDataFrame(communities_pd)

sentinel = (
    pagerank_df
    .join(
        bus_factor_sdf.select(
            col('package_name'),
            col('bus_factor'),
            col('recency_decay'),
            col('last_commit_date'),
            col('days_since_last_commit'),
            col('total_commits_12mo')
        ),
        col('name') == col('package_name'),
        'left'
    ).drop('package_name')
    .join(
        communities_sdf.select(col('id').alias('comm_id'), col('community_id')),
        col('id') == col('comm_id'),
        'left'
    ).drop('comm_id')
    # Fill defaults for packages with no commit data
    .withColumn('bus_factor',    coalesce(col('bus_factor').cast(DoubleType()),    lit(1.0)))
    .withColumn('recency_decay', coalesce(col('recency_decay').cast(DoubleType()), lit(0.01)))
    # Compute Sentinel Score
    .withColumn('sentinel_score',
        col('pagerank') * (1.0 / col('bus_factor')) * col('recency_decay')
    )
    # Risk tier classification
    .withColumn('risk_tier',
        when(col('sentinel_score') > 0.1,  lit('CRITICAL'))
        .when(col('sentinel_score') > 0.01, lit('HIGH'))
        .when(col('sentinel_score') > 0.001, lit('MEDIUM'))
        .otherwise(lit('LOW'))
    )
    .orderBy(col('sentinel_score').desc())
)

print('TOP 20 — Sentinel Score Leaderboard:')
sentinel.select(
    'name', 'sentinel_score', 'pagerank', 'bus_factor', 'recency_decay', 'risk_tier'
).show(20, truncate=False)

In [ ]:
sentinel_pd = sentinel.toPandas()
sentinel_pd.to_parquet(f'{OUT_DIR}/sentinel_scores.parquet', index=False)

print(f'Saved sentinel scores for {len(sentinel_pd):,} packages')
print('\nRisk tier distribution:')
print(sentinel_pd['risk_tier'].value_counts())

## 7 — Sanity checks

In [ ]:
# Check 1: React, lodash, express should be near the top of PageRank
print('CHECK 1 — PageRank for known packages:')
known = ['react', 'lodash', 'express', 'typescript', 'chalk']
print(pagerank_pd[pagerank_pd['name'].isin(known)][['name', 'pagerank']].to_string(index=False))

In [ ]:
# Check 2: A well-known popular package should have Bus Factor > 1
print('CHECK 2 — Bus Factor for well-maintained packages:')
well_maintained = ['react', 'typescript', 'webpack']
print(bus_factor_pd[bus_factor_pd['package_name'].isin(well_maintained)]
      [['package_name', 'bus_factor', 'last_commit_date']].to_string(index=False))

In [ ]:
# Check 3: Sentinel score for a known high-risk package should be near the top
print('CHECK 3 — Sentinel score spot-check:')
print(sentinel_pd.head(5)[['name', 'sentinel_score', 'pagerank', 'bus_factor', 'risk_tier']].to_string(index=False))
print('\n(manually verify: do these look like genuinely critical packages?)')

In [ ]:
print('Analytics complete. Outputs saved to data/processed/')
print('Next: open 04_enrichment.ipynb')